# Episode Data Inspector
Sanity check for collected demonstration data.
Reads one episode folder from `logs/` and plots key signals.

In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# Locate repo root relative to this notebook (assumed to be in analysis/)
NOTEBOOK_DIR = Path(os.path.abspath(''))
REPO_ROOT = NOTEBOOK_DIR.parent.parent
LOG_ROOT = REPO_ROOT / 'build/logs'

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Repo root:    {REPO_ROOT}')
print(f'Log root:     {LOG_ROOT}')
print(f'Exists:       {LOG_ROOT.exists()}')

Notebook dir: c:\Users\ceti\Documents\01_projects\teleop-simulator\analysis\notebooks
Repo root:    c:\Users\ceti\Documents\01_projects\teleop-simulator
Log root:     c:\Users\ceti\Documents\01_projects\teleop-simulator\build\logs
Exists:       True


## 1. List available episodes

In [11]:
episodes = sorted([d for d in LOG_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(episodes)} episode(s):")
for ep in episodes:
    files = list(ep.iterdir())
    print(f"  {ep.name}:  {[f.name for f in files]}")

Found 1 episode(s):
  000:  ['arm_left.csv', 'arm_left_meta.csv', 'arm_right.csv', 'arm_right_meta.csv', 'head.csv', 'head_meta.csv', 'images.hdf5', 'intention_log.csv', 'intention_log_meta.csv', 'scene.csv', 'scene_meta.csv']


## 2. Select episode to inspect

In [12]:
EPISODE = episodes[-1]  # change index to inspect a specific episode
print(f"Inspecting: {EPISODE}")

Inspecting: c:\Users\ceti\Documents\01_projects\teleop-simulator\build\logs\000


## 3. Load data

In [13]:
def load_arm(path):
    df = pd.read_csv(path, sep=';')
    # Reconstruct EE position from flat 4x4 column-major matrix
    # O_T_EE is column-major: translation is at indices [12,13,14]
    df['ee_x'] = df['O_T_EE_12']
    df['ee_y'] = df['O_T_EE_13']
    df['ee_z'] = df['O_T_EE_14']
    df['ee_cmd_x'] = df['O_T_EE_cmd_12']
    df['ee_cmd_y'] = df['O_T_EE_cmd_13']
    df['ee_cmd_z'] = df['O_T_EE_cmd_14']
    # Normalise time to start at 0
    df['t'] = df['time'] - df['time'].iloc[0]
    return df

left_path  = EPISODE / 'arm_left.csv'
right_path = EPISODE / 'arm_right.csv'
scene_path = EPISODE / 'scene.csv'
meta_left  = EPISODE / 'arm_left_meta.csv'

left  = load_arm(left_path)  if left_path.exists()  else None
right = load_arm(right_path) if right_path.exists() else None
scene = pd.read_csv(scene_path, sep=';') if scene_path.exists() else None
meta  = pd.read_csv(meta_left, sep=';')  if meta_left.exists()  else None

if scene is not None:
    scene['t'] = scene['time'] - scene['time'].iloc[0]

# Extract episode config from meta
cfg_row = meta[meta['event'] == 'episode_config'].iloc[0] if meta is not None else None
if cfg_row is not None:
    pick  = np.array([cfg_row['pick_x'],  cfg_row['pick_y'],  cfg_row['pick_z']])
    place = np.array([cfg_row['place_x'], cfg_row['place_y'], cfg_row['place_z']])
    mode  = 'unimanual' if int(cfg_row['mode']) == 0 else 'bimanual'
    print(f"Mode:  {mode}")
    print(f"Pick:  {pick}")
    print(f"Place: {place}")

print(f"Left arm rows:  {len(left) if left is not None else 'missing'}")
print(f"Right arm rows: {len(right) if right is not None else 'missing'}")
print(f"Scene rows:     {len(scene) if scene is not None else 'missing'}")
print(f"Duration:       {left['t'].iloc[-1]:.2f}s" if left is not None else '')

IndexError: single positional indexer is out-of-bounds

## 4. EE trajectory (base frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
labels = ['x', 'y', 'z']
cols   = ['ee_x', 'ee_y', 'ee_z']
cols_c = ['ee_cmd_x', 'ee_cmd_y', 'ee_cmd_z']

for i, ax in enumerate(axes):
    if left is not None:
        ax.plot(left['t'], left[cols[i]],   color='steelblue',  lw=1.2, label='left actual')
        ax.plot(left['t'], left[cols_c[i]], color='steelblue',  lw=0.7, ls='--', alpha=0.6, label='left cmd')
    if right is not None:
        ax.plot(right['t'], right[cols[i]],   color='tomato', lw=1.2, label='right actual')
        ax.plot(right['t'], right[cols_c[i]], color='tomato', lw=0.7, ls='--', alpha=0.6, label='right cmd')
    ax.set_ylabel(f'EE {labels[i]} [m]')
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

axes[-1].set_xlabel('time [s]')
fig.suptitle('End-effector trajectory (robot base frame)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Gripper width

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
if left is not None:
    ax.plot(left['t'], left['gripper_width'], color='steelblue', lw=1.5, label='left')
if right is not None:
    ax.plot(right['t'], right['gripper_width'], color='tomato', lw=1.5, label='right')
ax.set_ylabel('gripper width [m]')
ax.set_xlabel('time [s]')
ax.set_ylim(-0.005, 0.09)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.axhline(0.08, color='k', lw=0.5, ls='--', alpha=0.4)
ax.legend()
ax.grid(True, alpha=0.3)
fig.suptitle('Gripper width (0 = closed, 0.08 = fully open)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Object pose vs pick/place targets (world frame)

In [ ]:
if scene is not None and cfg_row is not None:
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    obj_cols = ['object_x', 'object_y', 'object_z']
    labels   = ['x', 'y', 'z']

    for i, ax in enumerate(axes):
        ax.plot(scene['t'], scene[obj_cols[i]], color='darkorange', lw=1.5, label='object')
        ax.axhline(pick[i],  color='green', lw=1.0, ls='--', label='pick target')
        ax.axhline(place[i], color='purple', lw=1.0, ls='--', label='place target')
        ax.set_ylabel(f'object {labels[i]} [m]')
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=8)

    axes[-1].set_xlabel('time [s]')
    fig.suptitle('Object position vs pick/place targets (world frame)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No scene.csv or episode config found — skipping object plot')

## 7. External forces (contact proxy)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
f_cols = [f'F_ext_{i}' for i in range(6)]
f_labels = ['fx', 'fy', 'fz', 'tx', 'ty', 'tz']

for df, name, color, ax in [(left, 'left', 'steelblue', axes[0]), (right, 'right', 'tomato', axes[1])]:
    if df is None:
        continue
    for col, label in zip(f_cols[:3], f_labels[:3]):  # forces only
        ax.plot(df['t'], df[col], lw=0.8, label=label)
    ax.set_ylabel(f'{name} F_ext [N]')
    ax.axhline(0, color='k', lw=0.5)
    ax.legend(fontsize=7, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('time [s]')
fig.suptitle('External forces (contact proxy)', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Joint positions

In [ ]:
q_cols = [f'q_{i}' for i in range(7)]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for df, name, color, ax in [(left, 'left', 'steelblue', axes[0]), (right, 'right', 'tomato', axes[1])]:
    if df is None:
        continue
    for j, col in enumerate(q_cols):
        ax.plot(df['t'], df[col], lw=0.8, label=f'q{j}')
    ax.set_ylabel(f'{name} q [rad]')
    ax.legend(fontsize=7, ncol=7)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('time [s]')
fig.suptitle('Joint positions', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Logging rate check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for df, name, ax in [(left, 'arm_left', axes[0]), (right, 'arm_right', axes[1])]:
    if df is None:
        continue
    dt = np.diff(df['time'].values) * 1000  # ms
    ax.hist(dt, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(np.mean(dt), color='red', lw=1.5, label=f'mean {np.mean(dt):.2f}ms ({1000/np.mean(dt):.0f}Hz)')
    ax.set_xlabel('dt [ms]')
    ax.set_ylabel('count')
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Logging rate distribution', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. 3D EE trajectory (world frame)
Applies fixed base transforms from robot_config.yaml

In [ ]:
from scipy.spatial.transform import Rotation as R

def base_transform(df, pos_base, quat_base_wxyz):
    """Transform EE positions from robot base frame to world frame."""
    q = np.array(quat_base_wxyz)  # w,x,y,z
    rot = R.from_quat([q[1], q[2], q[3], q[0]])  # scipy uses x,y,z,w
    p_base = np.array(pos_base)
    ee_base = df[['ee_x', 'ee_y', 'ee_z']].values
    ee_world = (rot.apply(ee_base) + p_base)
    return ee_world

# From robot_config.yaml
LEFT_BASE_POS  = [0, 0.4, 0.8]
LEFT_BASE_QUAT = [0.5, -0.5, 0.5, -0.5]   # w,x,y,z
RIGHT_BASE_POS  = [0, -0.4, 0.8]
RIGHT_BASE_QUAT = [0.5, 0.5, 0.5, 0.5]    # w,x,y,z

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

if left is not None:
    ee_l = base_transform(left, LEFT_BASE_POS, LEFT_BASE_QUAT)
    ax.plot(ee_l[:,0], ee_l[:,1], ee_l[:,2], color='steelblue', lw=1.2, label='left EE')
    ax.scatter(*ee_l[0], color='steelblue', s=40, marker='o')

if right is not None:
    ee_r = base_transform(right, RIGHT_BASE_POS, RIGHT_BASE_QUAT)
    ax.plot(ee_r[:,0], ee_r[:,1], ee_r[:,2], color='tomato', lw=1.2, label='right EE')
    ax.scatter(*ee_r[0], color='tomato', s=40, marker='o')

if cfg_row is not None:
    ax.scatter(*pick,  color='green',  s=80, marker='*', zorder=5, label='pick')
    ax.scatter(*place, color='purple', s=80, marker='*', zorder=5, label='place')

if scene is not None:
    ax.plot(scene['object_x'], scene['object_y'], scene['object_z'],
            color='darkorange', lw=1.0, ls='--', label='object')

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('z [m]')
ax.legend(fontsize=8)
ax.set_title('3D EE trajectory (world frame)', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Task completion heuristic
Simple check: did the object get close to the place target at any point?

In [ ]:
PLACE_THRESHOLD = 0.05  # metres

if scene is not None and cfg_row is not None:
    obj_pos = scene[['object_x', 'object_y', 'object_z']].values
    dist_to_place = np.linalg.norm(obj_pos - place, axis=1)
    dist_to_pick  = np.linalg.norm(obj_pos - pick,  axis=1)

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(scene['t'], dist_to_place, color='purple', lw=1.5, label='dist to place')
    ax.plot(scene['t'], dist_to_pick,  color='green',  lw=1.5, label='dist to pick')
    ax.axhline(PLACE_THRESHOLD, color='k', lw=1.0, ls='--', label=f'threshold ({PLACE_THRESHOLD}m)')
    ax.set_ylabel('distance [m]')
    ax.set_xlabel('time [s]')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    success = (dist_to_place < PLACE_THRESHOLD).any()
    min_dist = dist_to_place.min()
    ax.set_title(f'Object distance to targets — SUCCESS: {success} (min dist to place: {min_dist:.3f}m)',
                 fontweight='bold', color='green' if success else 'red')
    plt.tight_layout()
    plt.show()
else:
    print('No scene data — skipping completion check')